# Appendix A - Numerical Methods Behind SciBmad

This appendix collects several numerical ideas that appear throughout the tutorial, often without being named explicitly every time. The goal is not to replace a numerical analysis textbook, but to explain the computational backbone behind common SciBmad workflows: tracking particles, building maps, finding closed orbits, correcting errors, and fitting lattice parameters.

A useful way to read the main chapters is to keep the following hierarchy in mind:

$$
\text{lattice elements}
\quad \longrightarrow \quad
\text{tracking methods}
\quad \longrightarrow \quad
\text{maps and response matrices}
\quad \longrightarrow \quad
\text{analysis and solvers}.
$$

The high-level functions such as `track`, `twiss`, `find_closed_orbit`, `dynamic_aperture`, and the optimization routines all sit on top of this lower-level machinery.


## A.1 Tracking Methods

In SciBmad, a `tracking_method` specifies how a beamline element is traversed numerically. It is therefore more than a setting for long-term particle tracking. It defines the local map through an element, and that local map can later be used by many higher-level calculations.

For example, the same element-by-element propagation machinery can enter:

- particle tracking,
- closed-orbit finding,
- Twiss and normal-form calculations,
- dynamic aperture studies,
- radiation-damping studies,
- response-matrix construction,
- optimization and error fitting.

A tracking method may control the integration order, the number of integration steps, the longitudinal step size, fringe-field treatment, and optional effects such as radiation damping or fluctuations. Thus a lattice is not only a list of magnets and field strengths; it is also a prescription for how the equations of motion are evaluated through those magnets.

If an element has length $L$, a tracking method usually replaces the exact element map

$$
z_{\mathrm{out}} = M_L(z_{\mathrm{in}})
$$

with a numerical composition of smaller maps,

$$
M_L \approx M_{\Delta s_N} \circ M_{\Delta s_{N-1}} \circ \cdots \circ M_{\Delta s_1}.
$$

Here $z$ denotes the phase-space coordinates, and the step maps $M_{\Delta s_i}$ are chosen so that the approximation is accurate and, when appropriate, symplectic.


## A.2 Yoshida Symplectic Integration

For finite-length magnetic elements, the central integration method used in the SciBmad tutorial is the Yoshida symplectic integrator. SciBmad also has other tracking-method objects, but Yoshida is the main method to keep in mind when thinking about the default drift-kick style integration used for many beamline elements.

The basic idea is to split the Hamiltonian into simpler pieces. In an idealized form, suppose

$$
H(q,p) = T(p) + V(q),
$$

where $T$ generates drift-like motion and $V$ generates kick-like motion. The exact one-step map is formally

$$
\exp\left(h L_H\right)
=
\exp\left(h(L_T + L_V)\right),
$$

where $L_H$ is the Lie operator associated with the Hamiltonian $H$. Since the two pieces are usually easier to apply separately, a second-order symmetric splitting is

$$
S_2(h)
=
\exp\left(\frac{h}{2}L_T\right)
\exp\left(hL_V\right)
\exp\left(\frac{h}{2}L_T\right).
$$

This is the familiar drift-kick-drift structure. Yoshida's construction builds higher-order symplectic integrators by composing lower-order ones with carefully chosen coefficients. A fourth-order Yoshida composition can be written as

$$
S_4(h) = S_2(w_1 h) S_2(w_0 h) S_2(w_1 h),
$$

with

$$
w_1 = \frac{1}{2 - 2^{1/3}},
\qquad
w_0 = -\frac{2^{1/3}}{2 - 2^{1/3}}.
$$

The coefficients cancel lower-order error terms while preserving the symplectic structure, because a composition of symplectic maps is still symplectic.

This matters for storage-ring calculations. A non-symplectic method can introduce artificial damping, artificial excitation, or slow phase-space distortion over many turns. A symplectic method does not make the integration exact, but it keeps the long-term geometry of the motion much closer to the Hamiltonian system being modeled.

In practical SciBmad usage, parameters such as `order`, `n_steps`, and `ds_step` control the tradeoff between speed and accuracy. A larger number of steps or a higher order generally improves the element map, but increases the cost of tracking. Radiation damping and stochastic radiation can also be inserted into the tracking process; when these effects are enabled, the dynamics is no longer purely Hamiltonian, but the integration framework still organizes where those effects are applied.


## A.3 Optimization Methods

Many accelerator-design tasks can be written as least-squares problems. We choose a vector of adjustable variables

$$
u = (u_1, u_2, \ldots, u_n),
$$

and try to reduce a vector of residuals

$$
r(u) = (r_1(u), r_2(u), \ldots, r_m(u)).
$$

The variables might be quadrupole strengths, corrector kicks, sextupole strengths, RF parameters, or alignment errors. The residuals might be orbit errors, tune errors, Twiss mismatch, chromaticity errors, coupling terms, or measured-minus-modeled beam-position readings.

The local sensitivity of the residuals to the variables is the Jacobian, or response matrix,

$$
J_{ij} = \frac{\partial r_i}{\partial u_j}.
$$

Near the current parameter vector $u$, we linearize the residuals:

$$
r(u + \Delta u) \approx r(u) + J\Delta u.
$$

The Gauss-Newton step chooses $\Delta u$ by minimizing the linearized least-squares objective,

$$
\min_{\Delta u} \left\| r + J\Delta u \right\|_2^2.
$$

The corresponding normal equations are

$$
J^T J\Delta u = -J^T r.
$$

This method is powerful when the model is locally close to linear and the response matrix is well conditioned. In accelerator applications, however, response matrices are often nearly singular or strongly correlated. Damped least squares adds a regularization term:

$$
\left(J^T J + \lambda I\right)\Delta u = -J^T r,
$$

where $\lambda \ge 0$ is the damping parameter. Increasing $\lambda$ makes the step smaller and more stable; decreasing $\lambda$ makes the method closer to ordinary Gauss-Newton.

This is the numerical core behind many correction and fitting workflows:

$$
\text{compute residuals}
\quad \longrightarrow \quad
\text{compute response matrix}
\quad \longrightarrow \quad
\text{solve for correction}
\quad \longrightarrow \quad
\text{update lattice}.
$$

The response matrix can be computed by finite differences, automatic differentiation, or map-based methods such as GTPSA, depending on the problem and the available differentiability of the model.


## A.4 Newton Solver

Closed-orbit finding is a fixed-point problem. Let $M$ be the one-turn map of the ring. A closed orbit $z_{\mathrm{co}}$ satisfies

$$
M(z_{\mathrm{co}}) = z_{\mathrm{co}}.
$$

Equivalently, define

$$
F(z) = M(z) - z.
$$

Then the closed orbit is a root of

$$
F(z) = 0.
$$

Newton's method solves this nonlinear equation iteratively. At the current guess $z_k$, we linearize

$$
F(z_k + \Delta z)
\approx
F(z_k) + \frac{\partial F}{\partial z}\bigg|_{z_k}\Delta z.
$$

Setting this approximation to zero gives the Newton correction equation

$$
\frac{\partial F}{\partial z}\bigg|_{z_k}\Delta z = -F(z_k),
$$

followed by the update

$$
z_{k+1} = z_k + \Delta z.
$$

Since $F(z) = M(z) - z$, the Jacobian is

$$
\frac{\partial F}{\partial z}
=
\frac{\partial M}{\partial z} - I.
$$

Thus closed-orbit finding sits directly on top of tracking and differentiation. Tracking provides the one-turn map $M$, while differentiation provides the linearized map $\partial M/\partial z$. In practice, the derivative can come from automatic differentiation, finite differences, or GTPSA-based map construction.

This is why closed-orbit finding is sensitive to the underlying tracking method. If the element maps change because the integrator order, step size, radiation settings, or fringe-field model changes, then the one-turn map $M$ changes, and the closed orbit may change as well.


## A.5 Summary

The algorithms in this appendix play different roles:

- Tracking methods describe how particles and maps pass through elements.
- Yoshida integration provides a symplectic drift-kick framework for long-term Hamiltonian dynamics.
- Optimization methods convert design goals into residuals, response matrices, and correction steps.
- Newton solvers find fixed points such as closed orbits by repeatedly linearizing the one-turn map.

Together, these methods form the numerical layer underneath much of the tutorial. The user-facing SciBmad commands are compact, but they are built on this chain of tracking, differentiation, map construction, and linearized solving.
